<a href="https://colab.research.google.com/github/KULDEEP-PATIDAR/-Loan-Application-Predictor/blob/main/DT_undersample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns


In [3]:
!unzip "Training Data.csv.zip"

Archive:  Training Data.csv.zip
  inflating: Training Data.csv       


In [4]:
df=pd.read_csv('/content/Training Data.csv')

In [5]:

df.head()

,Id,Income,Age,Experience,Married/Single,House_Ownership,Car_Ownership,Profession,CITY,STATE,CURRENT_JOB_YRS,CURRENT_HOUSE_YRS,Risk_Flag
0,1,1303834,23,3,single,rented,no,Mechanical_engineer,Rewa,Madhya_Pradesh,3,13,0
1,2,7574516,40,10,single,rented,no,Software_Developer,Parbhani,Maharashtra,9,13,0
2,3,3991815,66,4,married,rented,no,Technical_writer,Alappuzha,Kerala,4,10,0
3,4,6256451,41,2,single,rented,yes,Software_Developer,Bhubaneswar,Odisha,2,12,1
4,5,5768871,47,11,single,rented,no,Civil_servant,Tiruchirappalli[10],Tamil_Nadu,3,14,1


In [6]:

df.shape

(252000, 13)

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 252000 entries, 0 to 251999
Data columns (total 13 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   Id                 252000 non-null  int64 
 1   Income             252000 non-null  int64 
 2   Age                252000 non-null  int64 
 3   Experience         252000 non-null  int64 
 4   Married/Single     252000 non-null  object
 5   House_Ownership    252000 non-null  object
 6   Car_Ownership      252000 non-null  object
 7   Profession         252000 non-null  object
 8   CITY               252000 non-null  object
 9   STATE              252000 non-null  object
 10  CURRENT_JOB_YRS    252000 non-null  int64 
 11  CURRENT_HOUSE_YRS  252000 non-null  int64 
 12  Risk_Flag          252000 non-null  int64 
dtypes: int64(7), object(6)
memory usage: 25.0+ MB


In [8]:
df['Risk_Flag'].value_counts()

,count
Risk_Flag,
0,221004
1,30996


In [9]:
df.isnull().sum()


,0
Id,0
Income,0
Age,0
Experience,0
Married/Single,0
House_Ownership,0
Car_Ownership,0
Profession,0
CITY,0
STATE,0


In [10]:

!pip install category-encoders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 1.2 MB/s eta 0:00:00


In [11]:

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
import category_encoders as ce
from sklearn import preprocessing

In [12]:

label_encoder = preprocessing.LabelEncoder()



In [13]:
label_encoder = LabelEncoder()

for col in ['Married/Single','Car_Ownership']:
    df[col] = label_encoder.fit_transform( df[col] )

In [14]:
pd.get_dummies(df, columns=["House_Ownership"])


,Id,Income,Age,Experience,Married/Single,Car_Ownership,Profession,CITY,STATE,CURRENT_JOB_YRS,CURRENT_HOUSE_YRS,Risk_Flag,House_Ownership_norent_noown,House_Ownership_owned,House_Ownership_rented
0,1,1303834,23,3,1,0,Mechanical_engineer,Rewa,Madhya_Pradesh,3,13,0,False,False,True
1,2,7574516,40,10,1,0,Software_Developer,Parbhani,Maharashtra,9,13,0,False,False,True
2,3,3991815,66,4,0,0,Technical_writer,Alappuzha,Kerala,4,10,0,False,False,True
3,4,6256451,41,2,1,1,Software_Developer,Bhubaneswar,Odisha,2,12,1,False,False,True
4,5,5768871,47,11,1,0,Civil_servant,Tiruchirappalli[10],Tamil_Nadu,3,14,1,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
251995,251996,8154883,43,13,1,0,Surgeon,Kolkata,West_Bengal,6,11,0,False,False,True
251996,251997,2843572,26,10,1,0,Army_officer,Rewa,Madhya_Pradesh,6,11,0,False,False,True
251997,251998,4522448,46,7,1,0,Design_Engineer,Kalyan-Dombivli,Maharashtra,7,12,0,False,False,True
251998,251999,6507128,45,0,1,0,Graphic_Designer,Pondicherry,Puducherry,0,10,0,False,False,True


In [15]:
onehot_encoder = OneHotEncoder(sparse_output = False)
df['House_Ownership'] = onehot_encoder.fit_transform(df['House_Ownership'].values.reshape(-1, 1) )


In [16]:
high_card_features = ['Profession', 'CITY', 'STATE']

count_encoder = ce.CountEncoder()


count_encoded = count_encoder.fit_transform( df[high_card_features] )
df = df.join(count_encoded.add_suffix("_count"))

In [17]:

df=df.drop(labels=['Profession', 'CITY', 'STATE'], axis=1)

In [18]:

df.head()


,Id,Income,Age,Experience,Married/Single,House_Ownership,Car_Ownership,CURRENT_JOB_YRS,CURRENT_HOUSE_YRS,Risk_Flag,Profession_count,CITY_count,STATE_count
0,1,1303834,23,3,1,0.0,0,3,13,0,5217,798,14122
1,2,7574516,40,10,1,0.0,0,9,13,0,5053,849,25562
2,3,3991815,66,4,0,0.0,0,4,10,0,5195,688,5805
3,4,6256451,41,2,1,0.0,1,2,12,1,5053,607,4658
4,5,5768871,47,11,1,0.0,0,3,14,1,4413,809,16537


In [19]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline

In [20]:
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

In [21]:
X = df.drop('Risk_Flag', axis=1)
y = df['Risk_Flag']

In [22]:
y.shape

(252000,)

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [24]:
dt_model = DecisionTreeClassifier(
    max_depth=10,
    min_samples_leaf=50,
    class_weight="balanced",
    random_state=42
)

In [25]:

dt_model.fit(X_train, y_train)


DecisionTreeClassifier(class_weight='balanced', max_depth=10,
                       min_samples_leaf=50, random_state=42)

In [26]:
idx_neg = np.where(y_test == 0)[0]
idx_pos = np.where(y_test == 1)[0]

np.random.seed(42)

eval_idx = np.concatenate([
    np.random.choice(idx_neg, 500, replace=False),
    np.random.choice(idx_pos, 500, replace=False)
])




In [27]:
idx_neg.shape

(44201,)

In [28]:
eval_idx.shape

(1000,)

In [29]:
X_eval = X_test.iloc[eval_idx]
y_eval = y_test.iloc[eval_idx]

In [30]:
X_eval.shape

(1000, 12)

In [31]:
y_eval.shape

(1000,)

In [32]:
y_pred = dt_model.predict(X_eval)

In [33]:
acc = accuracy_score(y_eval, y_pred)
prec = precision_score(y_eval, y_pred)
rec = recall_score(y_eval, y_pred)
f1 = f1_score(y_eval, y_pred)

cm = confusion_matrix(y_eval, y_pred)

print("Accuracy :", round(acc, 2))
print("Recall   :", round(rec, 2))
print("Precision:", round(prec, 2))
print("F1-score :", round(f1, 2))

print("\nConfusion Matrix:\n", cm)

Accuracy : 0.69
Recall   : 0.92
Precision: 0.63
F1-score : 0.75

Confusion Matrix:
 [[234 266]
 [ 42 458]]


In [34]:
pipeline = Pipeline(steps=[

    ('dt', DecisionTreeClassifier(class_weight="balanced", random_state=42))
])


In [35]:
param_dist = {
    'dt__max_depth': [None, 5,8, 10, 15, 20, 30],
    'dt__min_samples_split': [2, 5,8, 10, 15,20],
    'dt__min_samples_leaf': [1, 2, 5, 10],
    'dt__criterion': ['gini', 'entropy']
}

In [36]:
random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_dist,
    n_iter=20,
    scoring='f1',
    cv=5,
    random_state=42,
    n_jobs=-1
)

In [37]:
random_search.fit(X_train, y_train)

RandomizedSearchCV(cv=5,
                   estimator=Pipeline(steps=[('dt',
                                              DecisionTreeClassifier(class_weight='balanced',
                                                                     random_state=42))]),
                   n_iter=20, n_jobs=-1,
                   param_distributions={'dt__criterion': ['gini', 'entropy'],
                                        'dt__max_depth': [None, 5, 8, 10, 15,
                                                          20, 30],
                                        'dt__min_samples_leaf': [1, 2, 5, 10],
                                        'dt__min_samples_split': [2, 5, 8, 10,
                                                                  15, 20]},
                   random_state=42, scoring='f1')

In [38]:
print("Best Parameters:", random_search.best_params_)


Best Parameters: {'dt__min_samples_split': 10, 'dt__min_samples_leaf': 2, 'dt__max_depth': None, 'dt__criterion': 'gini'}


In [39]:
best_model = random_search.best_estimator_

y_pred_best = best_model.predict(X_eval)

In [40]:
acc = accuracy_score(y_eval, y_pred_best)
prec = precision_score(y_eval, y_pred_best)
rec = recall_score(y_eval, y_pred_best)
f1 = f1_score(y_eval, y_pred_best)

cm = confusion_matrix(y_eval, y_pred_best)

print("Accuracy :", round(acc, 2))
print("Recall   :", round(rec, 2))
print("Precision:", round(prec, 2))
print("F1-score :", round(f1, 2))

print("\nConfusion Matrix:\n", cm)

Accuracy : 0.71
Recall   : 0.57
Precision: 0.8
F1-score : 0.67

Confusion Matrix:
 [[428  72]
 [214 286]]


In [41]:
prec = precision_score(y_eval, y_pred_best,pos_label=0)
rec = recall_score(y_eval, y_pred_best,pos_label=0)
f1 = f1_score(y_eval, y_pred_best,pos_label=0)


print("Recall   :", round(rec, 2))
print("Precision:", round(prec, 2))
print("F1-score :", round(f1, 2))


Recall   : 0.86
Precision: 0.67
F1-score : 0.75


In [ ]:
y_prob = best_model.predict_proba(X_test)[:, 1]

In [ ]:
threshold = 0.30
y_pred_custom = (y_prob >= threshold).astype(int)

In [ ]:
print("Threshold:", threshold)
print("Precision:", precision_score(y_test, y_pred_custom))
print("Recall   :", recall_score(y_test, y_pred_custom))
print("F1 Score :", f1_score(y_test, y_pred_custom))

Threshold: 0.3
Precision: 0.13434144203374973
Recall   : 0.9888691724471689
F1 Score : 0.23654710683208244


In [ ]:
print("Accuracy :", accuracy_score(y_test, y_pred_best))
print("Precision:", precision_score(y_test, y_pred_best))
print("Recall   :", recall_score(y_test, y_pred_best))
print("F1 Score :", f1_score(y_test, y_pred_best))

print("\nDetailed Classification Report:\n")
print(classification_report(y_test, y_pred_best))

Accuracy : 0.5158531746031746
Precision: 0.16030904747685876
Recall   : 0.6928536860783997
F1 Score : 0.26037404140523174

Detailed Classification Report:

              precision    recall  f1-score   support

           0       0.92      0.49      0.64     44201
           1       0.16      0.69      0.26      6199

    accuracy                           0.52     50400
   macro avg       0.54      0.59      0.45     50400
weighted avg       0.83      0.52      0.59     50400



In [ ]:
from sklearn.metrics import confusion_matrix

In [ ]:
cm_best = confusion_matrix(y_test, y_pred_best)
print("Confusion Matrix (Best Model):\n", cm_best)

Confusion Matrix (Best Model):
 [[21704 22497]
 [ 1904  4295]]
